# Advanced 2-Stage Pipeline with UMAP

This notebook implements a sophisticated hierarchical classification approach:

1. **Data Preparation**: Load cleaned data and create advanced features
2. **Dimensionality Reduction**: Use UMAP to capture complex patterns
3. **Stage 1 - F-Detection**: Binary classifier to identify students at risk of failing (F grade)
4. **Stage 2 - Grade Classification**: 4-class classifier (D, C, B, A) for students who passed
5. **Hybrid Inference**: Combine predictions from both stages for final results
6. **Feature Analysis**: Understand which factors drive each decision stage

**Rationale**: Many students pass (grades D-A), so a dedicated F-detector followed by fine-grained classification improves performance on minority class (F).

## Imports & Setup

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay

from xgboost import XGBClassifier
from umap import UMAP

# Configuration
RANDOM_STATE = 42
plt.rcParams['figure.dpi'] = 110
sns.set_style('whitegrid')

print("✓ All imports successful")
print(f"  UMAP version installed")

## Load Data

In [ ]:
# Load cleaned YRBS data
df = pd.read_csv('../data/cleaned_yrbs_data.csv')

print(f"Data shape: {df.shape}")
print(f"\nTarget variable (grade) distribution:")
grade_counts = df['grade'].value_counts().sort_index()
print(grade_counts)
print(f"\nClass proportions:")
print((grade_counts / grade_counts.sum()).round(3))

df.head()

## Feature Engineering

In [ ]:
# Create comprehensive feature engineering
print("Creating engineered features...")

# 1. Health & Lifestyle
health_cols = ['hours_of_sleep', 'exercise', 'mental_health', 'ADHD']
df['health_score'] = df[health_cols].sum(axis=1)
df['health_risk'] = ((df['hours_of_sleep'] < 7) | (df['mental_health'] > 2) | (df['ADHD'] > 0)).astype(int)
df['sleep_exercise_interaction'] = df['hours_of_sleep'] * df['exercise']
df['mental_adhd_interaction'] = df['mental_health'] * df['ADHD']

# 2. Alcohol Use
alcohol_cols = ['first_alcohol', 'alcohol_frequency', 'hardcore_alcohol_frequency', 'drinking_in_a_row']
df['alcohol_exposure'] = df[alcohol_cols].sum(axis=1)
df['alcohol_progression'] = df['alcohol_frequency'] - df['first_alcohol']
df['high_alcohol_use'] = ((df['alcohol_frequency'] > 0) | (df['hardcore_alcohol_frequency'] > 0)).astype(int)

# 3. Family Environment
family_cols = ['parent_emotional_abuse', 'parent_physical_abuse', 'parent_abuse_parent', 
               'parent_uses_alcohol', 'parent_mental_illness', 'incarcerated_parent', 'parent_monitoring']
df['family_adversity_score'] = df[family_cols[:-1]].sum(axis=1)
df['family_protective_score'] = df['parent_monitoring'] + df['have_friend']
df['family_risk_ratio'] = df['family_adversity_score'] / (df['family_protective_score'] + 1)

# 4. School Environment
school_cols = ['school_safety_concern', 'threatened_at_school', 'school_fight_count', 
               'school_racism', 'school_bullying', 'unfair_discipline_school']
df['school_stress_score'] = df[school_cols].sum(axis=1)
df['bullying_exposure'] = ((df['school_bullying'] > 0) | (df.get('cyber_bullying', pd.Series([0]*len(df))) > 0)).astype(int)
df['school_violence_index'] = df['school_safety_concern'] + df['threatened_at_school'] + df['school_fight_count']

# 5. Social & Peer
df['social_engagement'] = df['social_media'] + df['have_friend']
df['social_risk'] = (df['social_media'] > 5).astype(int)
df['peer_support'] = df['have_friend']

# 6. Violence & Safety
violence_cols = ['fight_count', 'neighborhood_violence', 'sexual_violence', 'ride_drinking_driver']
df['violence_exposure_score'] = df[violence_cols].sum(axis=1)
df['personal_violence'] = df['fight_count'] + df['sexual_violence']
df['environmental_violence'] = df['neighborhood_violence'] + df['ride_drinking_driver']

# 7. Demographic & Cross-domain
df['age_squared'] = df['age'] ** 2
df['adolescent_risk'] = ((df['age'] >= 14) & (df['age'] <= 17)).astype(int)
df['health_substance_risk'] = df['health_risk'] * df['high_alcohol_use']
df['family_school_stress'] = df['family_adversity_score'] * df['school_stress_score']
df['social_violence_link'] = df['social_risk'] * df['violence_exposure_score']

print(f"✓ Feature engineering complete")
print(f"  Original features: 34")
print(f"  Engineered features: {df.shape[1] - 34}")
print(f"  Total features: {df.shape[1]}")

## Preprocessing & Dimensionality Reduction (UMAP)

In [ ]:
# Prepare features for preprocessing
X = df.drop(columns=['grade'])
y = df['grade'].astype(int)

# Define categorical and numeric features
categorical_features = ['gender', 'education_level', 'considered_suicide', 'sleep_place']
numeric_features = [col for col in X.columns if col not in categorical_features]

print(f"Categorical features: {len(categorical_features)}")
print(f"Numeric features: {len(numeric_features)}")

# Create preprocessor (StandardScaler + OneHotEncoder)
print("\nCreating preprocessor...")
preprocessor = ColumnTransformer(transformers=[
    ('num', StandardScaler(), numeric_features),
    ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_features)
])

X_preprocessed = preprocessor.fit_transform(X)
print(f"✓ Preprocessed feature shape: {X_preprocessed.shape}")
print(f"  (standardized numeric + one-hot encoded categorical)")

In [ ]:
# Apply UMAP for dimensionality reduction
print("\nApplying UMAP dimensionality reduction...")
reducer = UMAP(
    n_neighbors=15,
    min_dist=0.1,
    n_components=2,
    random_state=RANDOM_STATE,
    verbose=0
)
umap_features = reducer.fit_transform(X_preprocessed)
print(f"✓ UMAP features shape: {umap_features.shape}")
print(f"  (Reduced {X_preprocessed.shape[1]} dimensions → 2 UMAP dimensions)")

# Combine original preprocessed features with UMAP features
X_enhanced = np.hstack([X_preprocessed, umap_features])
print(f"\n✓ Enhanced feature set: {X_enhanced.shape}")
print(f"  (Original {X_preprocessed.shape[1]} + UMAP 2 = {X_enhanced.shape[1]} total)")

## Stage 1: F-Detector (Binary Classifier)

In [ ]:
# Prepare Stage 1: Binary classification for F grade detection
print("\n" + "="*70)
print("STAGE 1: F-DETECTOR (Binary Classification)")
print("="*70)

# Create binary target: 1 = Failed (F), 0 = Passed (D-A)
y_stage1 = (y == 0).astype(int)  # 1 if grade is F (0), else 0

print(f"\nClass distribution:")
print(f"  Passed (0): {(y_stage1 == 0).sum()} students")
print(f"  Failed (1): {(y_stage1 == 1).sum()} students")
print(f"  Imbalance ratio: {(y_stage1 == 0).sum() / (y_stage1 == 1).sum():.2f}:1")

# Train-test split for Stage 1
X_train1, X_test1, y_train1, y_test1 = train_test_split(
    X_enhanced, y_stage1, test_size=0.2, random_state=RANDOM_STATE, stratify=y_stage1
)

print(f"\nTrain set: {X_train1.shape[0]:,} samples")
print(f"Test set:  {X_test1.shape[0]:,} samples")

In [ ]:
# Train Stage 1 model with SMOTE-like weights
print("\nTraining Stage 1 F-Detector model...")
model_f_detector = XGBClassifier(
    n_estimators=1200,
    learning_rate=0.02,
    max_depth=5,
    scale_pos_weight=10,  # Weight minority class (F) more heavily
    random_state=RANDOM_STATE,
    verbosity=0
)
model_f_detector.fit(X_train1, y_train1)

# Evaluate Stage 1
y_pred_stage1_train = model_f_detector.predict(X_train1)
y_pred_stage1_test = model_f_detector.predict(X_test1)

train_acc_s1 = accuracy_score(y_train1, y_pred_stage1_train)
test_acc_s1 = accuracy_score(y_test1, y_pred_stage1_test)

print(f"✓ Stage 1 model trained")
print(f"  Train accuracy: {train_acc_s1:.4f}")
print(f"  Test accuracy:  {test_acc_s1:.4f}")
print(f"\nClassification Report (Stage 1):")
print(classification_report(y_test1, y_pred_stage1_test, target_names=['Passed', 'Failed (F)'], zero_division=0))

## Stage 2: Grade Classifier (D, C, B, A)

In [ ]:
# Prepare Stage 2: 4-class classification for passing grades
print("\n" + "="*70)
print("STAGE 2: GRADE CLASSIFIER (D, C, B, A)")
print("="*70)

# Select only students who passed (grade > 0)
pass_mask = (y > 0)
X_pass = X_enhanced[pass_mask]

# Map grades 1-4 to 0-3 for Stage 2 classification
grade_4group_map = {1: 0, 2: 1, 3: 2, 4: 3}  # D, C, B, A
y_stage2_raw = y[pass_mask].map(grade_4group_map)

print(f"\nClass distribution (among passing students):")
for grade_code, grade_name in enumerate(['D', 'C', 'B', 'A']):
    count = (y_stage2_raw == grade_code).sum()
    pct = count / len(y_stage2_raw) * 100
    print(f"  {grade_name}: {count:,} students ({pct:.1f}%)")

# Train-test split for Stage 2
X_train2, X_test2, y_train2, y_test2 = train_test_split(
    X_pass, y_stage2_raw, test_size=0.2, random_state=RANDOM_STATE, stratify=y_stage2_raw
)

print(f"\nTrain set: {X_train2.shape[0]:,} samples")
print(f"Test set:  {X_test2.shape[0]:,} samples")

In [ ]:
# Train Stage 2 model
print("\nTraining Stage 2 Grade Classifier model...")
model_stage2 = XGBClassifier(
    n_estimators=1200,
    learning_rate=0.02,
    max_depth=4,
    objective='multi:softprob',
    num_class=4,
    random_state=RANDOM_STATE,
    verbosity=0
)
model_stage2.fit(X_train2, y_train2)

# Evaluate Stage 2
y_pred_stage2_train = model_stage2.predict(X_train2)
y_pred_stage2_test = model_stage2.predict(X_test2)

train_acc_s2 = accuracy_score(y_train2, y_pred_stage2_train)
test_acc_s2 = accuracy_score(y_test2, y_pred_stage2_test)

print(f"✓ Stage 2 model trained")
print(f"  Train accuracy: {train_acc_s2:.4f}")
print(f"  Test accuracy:  {test_acc_s2:.4f}")
print(f"\nClassification Report (Stage 2):")
print(classification_report(y_test2, y_pred_stage2_test, target_names=['D', 'C', 'B', 'A'], zero_division=0))

## Hybrid Inference: Combine Both Stages

In [ ]:
# Hybrid inference logic: Combine Stage 1 and Stage 2 predictions
print("\n" + "="*70)
print("HYBRID INFERENCE: Combining Stage 1 and Stage 2")
print("="*70)

print("\nInference Logic:")
print("  IF Stage 1 predicts F (failed) → Final grade = F")
print("  ELSE → Use Stage 2 prediction for D/C/B/A")

# Get Stage 1 predictions for test set
stage1_preds_all = model_f_detector.predict(X_enhanced[~pass_mask | (pass_mask)])

# Map full test set predictions
final_preds_full = []
all_indices = np.arange(len(X_enhanced))

# Process all samples
for i in all_indices:
    if i in all_indices[~pass_mask]:  # Student failed according to Stage 1 threshold
        final_preds_full.append(0)  # F grade
    else:
        # Find corresponding index in X_pass
        pass_idx = np.where(all_indices[pass_mask] == i)[0]
        if len(pass_idx) > 0:
            stage2_pred = model_stage2.predict(X_pass[pass_idx[0]].reshape(1, -1))[0]
            final_preds_full.append(stage2_pred + 1)  # Map 0-3 back to 1-4

# For cleaner evaluation, use test set indices
# Stage 1 predictions on test portion
y_actual_stage1 = y[y_test1.index.intersection(y.index)]

# Simpler approach: test the full pipeline
final_preds = []
stage1_preds_test = model_f_detector.predict(X_test1)

for i in range(len(stage1_preds_test)):
    if stage1_preds_test[i] == 1:  # Predicted F
        final_preds.append(0)
    else:  # Predicted to pass, use Stage 2
        # Find this sample in X_pass
        # Map test indices to enhanced indices
        sample = X_test1[i].reshape(1, -1)
        stage2_pred = model_stage2.predict(sample)[0]
        final_preds.append(stage2_pred + 1)  # Remap to 1-4

# Get actual labels for test set from Stage 1 perspective
y_actual_test = y[y_test1.index].values

# Calculate accuracy
hybrid_acc = accuracy_score(y_actual_test, final_preds)

print(f"\n✓ Hybrid inference complete")
print(f"\nOverall Accuracy (Hybrid Pipeline): {hybrid_acc:.4f}")
print(f"\nDetailed Classification Report:")
print(classification_report(y_actual_test, final_preds, target_names=['F', 'D', 'C', 'B', 'A'], zero_division=0))

In [ ]:
# Confusion matrix for hybrid pipeline
cm_hybrid = confusion_matrix(y_actual_test, final_preds)
fig, ax = plt.subplots(figsize=(8, 7))
ConfusionMatrixDisplay(
    confusion_matrix=cm_hybrid,
    display_labels=['F', 'D', 'C', 'B', 'A'],
).plot(ax=ax, colorbar=False, cmap='Blues')
ax.set_title('2-Stage Hybrid Pipeline - Confusion Matrix', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## Feature Importance Analysis

In [ ]:
# Stage 1 Feature Importance (F-Detector)
print("\n" + "="*70)
print("STAGE 1: F-DETECTOR FEATURE IMPORTANCE")
print("="*70)

# Get feature names
num_names = numeric_features
cat_names = preprocessor.named_transformers_['cat'].get_feature_names_out(categorical_features).tolist()
all_feature_names = num_names + cat_names + ['UMAP_1', 'UMAP_2']

# Get importances
importances_s1 = model_f_detector.feature_importances_

if len(all_feature_names) == len(importances_s1):
    feat_import_s1_df = pd.DataFrame({
        'Feature': all_feature_names,
        'Importance': importances_s1
    }).sort_values(by='Importance', ascending=False)

    print("\nTop 15 Features for F-Detection:")
    print(feat_import_s1_df.head(15).to_string(index=False))

    fig, ax = plt.subplots(figsize=(12, 8))
    sns.barplot(x='Importance', y='Feature', data=feat_import_s1_df.head(15), palette='Reds_r', ax=ax)
    ax.set_title('Stage 1: Top 15 Features for F-Detection', fontsize=14, fontweight='bold')
    ax.grid(axis='x', linestyle='--', alpha=0.5)
    plt.tight_layout()
    plt.show()
else:
    print(f"⚠ Feature dimension mismatch: {len(all_feature_names)} names vs {len(importances_s1)} importances")

In [ ]:
# Stage 2 Feature Importance (Grade Classifier)
print("\n" + "="*70)
print("STAGE 2: GRADE CLASSIFIER FEATURE IMPORTANCE")
print("="*70)

importances_s2 = model_stage2.feature_importances_

if len(all_feature_names) == len(importances_s2):
    feat_import_s2_df = pd.DataFrame({
        'Feature': all_feature_names,
        'Importance': importances_s2
    }).sort_values(by='Importance', ascending=False)

    print("\nTop 15 Features for Grade Classification (D/C/B/A):")
    print(feat_import_s2_df.head(15).to_string(index=False))

    fig, ax = plt.subplots(figsize=(12, 8))
    sns.barplot(x='Importance', y='Feature', data=feat_import_s2_df.head(15), palette='viridis', ax=ax)
    ax.set_title('Stage 2: Top 15 Features for Grade Classification (D/C/B/A)', fontsize=14, fontweight='bold')
    ax.grid(axis='x', linestyle='--', alpha=0.5)
    plt.tight_layout()
    plt.show()
else:
    print(f"⚠ Feature dimension mismatch: {len(all_feature_names)} names vs {len(importances_s2)} importances")

## Results Summary & Export

In [ ]:
# Create results DataFrame
results_df = pd.DataFrame({
    'Actual': y_actual_test,
    'Predicted': final_preds
})

# Export results
results_df.to_csv('../results/2-stages_results.csv', index=False)
print("✓ Results saved to: ../results/2-stages_results.csv")

# Export feature importances
feat_import_s1_df.to_csv('../results/feature_importance_stage1_f_detector.csv', index=False)
feat_import_s2_df.to_csv('../results/feature_importance_stage2_grades.csv', index=False)
print("✓ Feature importances saved")

print(f"\n{'='*70}")
print("PIPELINE SUMMARY")
print(f"{'='*70}")
print(f"\nStage 1 (F-Detector):")
print(f"  - Accuracy: {test_acc_s1:.4f}")
print(f"\nStage 2 (Grade Classifier):")
print(f"  - Accuracy: {test_acc_s2:.4f}")
print(f"\nHybrid Pipeline (Combined):")
print(f"  - Final Accuracy: {hybrid_acc:.4f}")
print(f"  - Test samples: {len(y_actual_test)}")
print(f"\nApproach: Binary F-detection → 4-class grade classification for passers")